# Notebook 3.5 — Downstream **classification** (intuitive variant)

Same business question as [Notebook 3](03_training_and_evaluation.ipynb), but with an easier-to-read task:

**fine-tune YOLOv8s-cls** on **real-only** vs **real + synthetic**, evaluate on the **same held-out real test**.

With default knobs we **oversample** rare classes in train so SGD does not collapse to always guessing `scene` (460 vs ~14–75 images per class before balancing).

### Detection vs classification

| | Notebook 3 (detection) | This notebook (classification) |
|---|---|---|
| **Question** | *Where* is each object + *what* is it? | *What type of image is this?* (one label per image) |
| **Labels** | Bounding boxes from Mapillary GT / YOLO-World | Image **tag** from NB2 (`scene`, `traffic_cone`, `ground_animal`) |
| **Main metrics** | mAP50, AP50 per class | Top-1 accuracy, F1 / recall per class |
| **Why use this** | Matches production ADAS (boxes matter) | Easier to explain in a workshop |

Depends on [02_batch_dataset_generation.ipynb](02_batch_dataset_generation.ipynb) export under `outputs/<dataset>/nb2/`.

---
## What are we measuring? (classification metrics cheat sheet)

Each test image has **one true label** (its NB2 `tag`). The model outputs **one predicted label**.

| Term | Plain English |
|------|----------------|
| **Top-1 accuracy** | Fraction of test images where the **top prediction** equals the true label. “How often are we exactly right?” |
| **Precision** (per class) | When the model says “traffic_cone”, how often is it actually a cone image? |
| **Recall** (per class) | Of all true cone images, how many did we label as cone? |
| **F1** (per class) | Harmonic mean of precision and recall — balances “don’t cry wolf” vs “don’t miss rares”. |
| **Macro F1** | Average F1 across classes (each class weighted equally). Useful when classes are imbalanced. |

**Contrast with Notebook 3:** detection AP50 also requires drawing a box in roughly the right place. Classification only checks the **image-level** label — so scores are often **higher** and easier to interpret, but they **ignore localization**.

**How to read your table:** compare `real_synth` vs `real_only` on **rare-class F1 / recall** (`traffic_cone`, `ground_animal`). `scene` is the easy majority class.

---
## 0. Setup

In [ ]:
import sys
from pathlib import Path

%matplotlib inline

def _find_project_root() -> Path:
    here = Path.cwd().resolve()
    search = [here, *here.parents]
    for base in list(search):
        nested = base / "implementations" / "edge_case_image_generation"
        if nested.is_dir():
            search.append(nested)
    for base in search:
        if (base / "src" / "edgecase_synthesis").is_dir() and (base / "configs").is_dir():
            return base
    raise FileNotFoundError("Could not find edge_case_image_generation root")


PROJECT_ROOT = _find_project_root()
sys.path.insert(0, str(PROJECT_ROOT / "src"))
print("PROJECT_ROOT =", PROJECT_ROOT)

---
## 1. Knobs

Classification uses **image tags** from the NB2 manifest — not bounding boxes.

**Imbalance note:** your train split is ~460 `scene` vs ~14–75 rares. Without `BALANCE_TRAIN`, even fine-tuning usually learns “always predict `scene`” (60% test accuracy, 0 rare F1). Oversampling fixes that better than swapping to a bigger backbone alone.

**Why not YOLO-World + custom layers here?** YOLO-World is an open-vocabulary **detector** (boxes + text queries). For whole-image tags, a cls head + balanced sampling is simpler and matches the task. Production ADAS still belongs in **Notebook 3**.

In [ ]:
import os

os.environ.setdefault("HF_HUB_DISABLE_XET", "1")

from edgecase_synthesis.config import load_config
from edgecase_synthesis.classification_train import load_manifest

# --- learner knobs ---------------------------------------------------------
DATASET = "mapillary_vistas"
HARDWARE = "gpu_l4"  # or "cpu" for a tiny smoke run

# Image-level labels from NB2 stratified tags.
CLASS_NAMES = ["scene", "traffic_cone", "ground_animal"]

MODEL = "yolov8s-cls.pt"   # yolov8n-cls.pt for a faster/smokier run
BALANCE_TRAIN = True       # oversample rare train folders → match scene count
EPOCHS = 40
IMGSZ = 224                # cls default; detection NB3 uses 640
BATCH = 32
PATIENCE = 15
SEED = 42

NB2_DIR_NAME = "nb2"
# ---------------------------------------------------------------------------

cfg = load_config(
    start=PROJECT_ROOT,
    overrides=[f"dataset_name={DATASET}", f"hardware={HARDWARE}"],
)
nb2_dir = Path(cfg.paths.outputs_dir) / NB2_DIR_NAME
out_dir = Path(cfg.paths.outputs_dir) / "nb3_5"
out_dir.mkdir(parents=True, exist_ok=True)

train_manifest_path = nb2_dir / "train_manifest.json"
test_manifest_path = nb2_dir / "test_manifest.json"
if not train_manifest_path.exists() or not test_manifest_path.exists():
    raise FileNotFoundError(
        f"Missing NB2 export under {nb2_dir}. "
        "Run Notebook 2 first (train_manifest.json / test_manifest.json)."
    )

train_manifest = load_manifest(train_manifest_path)
test_manifest = load_manifest(test_manifest_path)
device = str(cfg.hardware.get("device", "cpu"))

print(f"Dataset:   {cfg.dataset_name}")
print(f"Hardware:  {cfg.hardware.name}  device={device}")
print(f"NB2:       {nb2_dir}")
print(f"NB3.5 out: {out_dir}")
print(f"Train rows: {len(train_manifest)}  Test rows: {len(test_manifest)}")
print(f"Classes:   {CLASS_NAMES}")
print(f"Train:     {MODEL}  balance={BALANCE_TRAIN}  epochs={EPOCHS}  imgsz={IMGSZ}  batch={BATCH}")

---
## 2. Build two classification datasets

| Run | Train images | Val / test |
|-----|--------------|------------|
| **real_only** | real train rows | real test (Mapillary tags) |
| **real_synth** | real train + accepted synth | same real test |

Folder layout: `train/<class>/image.jpg` — standard for Ultralytics classification.

In [ ]:
from edgecase_synthesis.classification_train import build_cls_dataset

ds_root = out_dir / "datasets"

root_real, tr_real, va_real = build_cls_dataset(
    train_manifest=train_manifest,
    test_manifest=test_manifest,
    out_dir=ds_root / "real_only",
    class_names=CLASS_NAMES,
    include_synthetic=False,
    balance_train=BALANCE_TRAIN,
    dataset_name="real_only",
)
root_both, tr_both, va_both = build_cls_dataset(
    train_manifest=train_manifest,
    test_manifest=test_manifest,
    out_dir=ds_root / "real_synth",
    class_names=CLASS_NAMES,
    include_synthetic=True,
    balance_train=BALANCE_TRAIN,
    dataset_name="real_synth",
)


def _print_stats(title, train_s, val_s):
    print(title)
    print(
        f"  train: images={train_s.n_images}  real={train_s.n_real}  "
        f"synth={train_s.n_synthetic}  per_class={train_s.images_per_class}"
    )
    print(f"  val:   images={val_s.n_images}  per_class={val_s.images_per_class}")
    if train_s.n_skipped:
        print(f"  skipped rows (unknown tag): {train_s.n_skipped}")


_print_stats("real_only", tr_real, va_real)
_print_stats("real_synth", tr_both, va_both)
if BALANCE_TRAIN:
    print("(train per_class counts are AFTER oversampling — expect ~460 each)")
print("dataset:", root_real)
print("dataset:", root_both)

---
## 3. Fine-tune A: real only

In [ ]:
from edgecase_synthesis.classification_train import train_classifier

runs_dir = out_dir / "runs"
run_real = train_classifier(
    root_real,
    name="real_only",
    project_dir=runs_dir,
    class_names=CLASS_NAMES,
    model_name=MODEL,
    epochs=EPOCHS,
    imgsz=IMGSZ,
    batch=BATCH,
    device=device,
    seed=SEED,
    patience=PATIENCE,
)
print("best weights:", run_real.weights)
print("metrics:", run_real.metrics)

---
## 4. Fine-tune B: real + synthetic

In [ ]:
run_synth = train_classifier(
    root_both,
    name="real_synth",
    project_dir=runs_dir,
    class_names=CLASS_NAMES,
    model_name=MODEL,
    epochs=EPOCHS,
    imgsz=IMGSZ,
    batch=BATCH,
    device=device,
    seed=SEED,
    patience=PATIENCE,
)
print("best weights:", run_synth.weights)
print("metrics:", run_synth.metrics)

---
## 5. Compare on held-out real test

Look at **rare-class F1 / recall** — accuracy alone can look good because `scene` dominates.

In [ ]:
import matplotlib.pyplot as plt

from edgecase_synthesis.classification_train import (
    metrics_table,
    plot_confusion_matrix,
    plot_metrics_comparison,
)
from edgecase_synthesis.eda import write_json

runs = [run_real, run_synth]
table = metrics_table(runs)
print(f"{'run':12s}  {'top1_acc':>8s}  {'macro_f1':>8s}", end="")
for cls in CLASS_NAMES:
    print(f"  {('F1 '+cls):>16s}", end="")
print()
for row in table:
    print(
        f"{row['run']:12s}  {float(row['top1_acc'] or 0):8.3f}  {float(row['macro_f1'] or 0):8.3f}",
        end="",
    )
    for cls in CLASS_NAMES:
        print(f"  {float(row.get(f'f1_{cls}') or 0):16.3f}", end="")
    print()

for run in runs:
    print(f"\n{run.name} confusion (pred):", run.metrics.get("confusion", {}).get("pred"))

write_json(out_dir / "comparison.json", table)
fig, _ = plot_metrics_comparison(
    runs,
    class_names=CLASS_NAMES,
    title="Real-only vs real+synth (held-out real test)",
)
fig.savefig(out_dir / "comparison.png", dpi=120, bbox_inches="tight")
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
for ax, run in zip(axes, runs):
    plot_confusion_matrix(run.metrics, class_names=CLASS_NAMES, title=run.name, ax=ax)
plt.tight_layout()
fig.savefig(out_dir / "confusion_compare.png", dpi=120, bbox_inches="tight")
plt.show()
print("Saved:", out_dir / "comparison.png", "and", out_dir / "confusion_compare.png")

---
## 6. Qualitative gallery

Side-by-side **predicted class** on real test images (rare tags first).

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image

from edgecase_synthesis.classification_train import predict_gallery

rare_test = [
    Path(r["path"])
    for r in test_manifest
    if r.get("split") != "synthetic" and r.get("tag") in ("traffic_cone", "ground_animal")
]
scene_test = [
    Path(r["path"])
    for r in test_manifest
    if r.get("split") != "synthetic" and r.get("tag") == "scene"
]
gallery_paths = (rare_test + scene_test)[:8]
print(f"gallery images: {len(gallery_paths)}")

gal_real = predict_gallery(
    run_real.weights,
    gallery_paths,
    class_names=CLASS_NAMES,
    out_dir=out_dir / "gallery_real_only",
    device=device,
)
gal_synth = predict_gallery(
    run_synth.weights,
    gallery_paths,
    class_names=CLASS_NAMES,
    out_dir=out_dir / "gallery_real_synth",
    device=device,
)

n = min(len(gal_real), len(gal_synth), 4)
if n:
    fig, axes = plt.subplots(n, 2, figsize=(10, 3 * n))
    if n == 1:
        axes = axes.reshape(1, 2)
    for i in range(n):
        axes[i, 0].imshow(Image.open(gal_real[i]))
        axes[i, 0].set_title(gal_real[i].name)
        axes[i, 0].axis("off")
        axes[i, 1].imshow(Image.open(gal_synth[i]))
        axes[i, 1].set_title(gal_synth[i].name)
        axes[i, 1].axis("off")
    fig.suptitle("Classification predictions (filename = pred + confidence)", y=1.01)
    plt.tight_layout()
    gallery_path = out_dir / "gallery_compare.png"
    fig.savefig(gallery_path, dpi=120, bbox_inches="tight")
    plt.show()
    print("Saved:", gallery_path)
else:
    print("No gallery images produced.")

---
## Wrap-up

| If you see… | Likely takeaway |
|-------------|-----------------|
| **real_synth F1 ↑** on rares | Synth helped the model recognize tail classes |
| **high accuracy, flat rare F1** | Model learned to always guess `scene` (majority class) |
| **cls ↑ but det flat** | Synth teaches *presence* but boxes are still noisy (see NB3) |

Artifacts: `outputs/<dataset>/nb3_5/` (`datasets/`, `runs/`, `comparison.json`, galleries).

For production ADAS, prefer **Notebook 3** (boxes). Use this notebook when explaining *whether synth changed what the model sees*.